In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [19]:
df = pd.read_csv("data.csv")
df.head(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [20]:
df.dtypes

id                           int64
diagnosis                      str
radius_mean                float64
texture_mean               float64
perimeter_mean             float64
area_mean                  float64
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean     float64
radius_se                  float64
texture_se                 float64
perimeter_se               float64
area_se                    float64
smoothness_se              float64
compactness_se             float64
concavity_se               float64
concave points_se          float64
symmetry_se                float64
fractal_dimension_se       float64
radius_worst               float64
texture_worst              float64
perimeter_worst            float64
area_worst                 float64
smoothness_worst           float64
compactness_worst          float64
concavity_worst     

In [21]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='str')

In [22]:
df.isna().mean()
## la colonne "Unnamed" ne contient aucune valeur, je vais la supprimer

id                         0.0
diagnosis                  0.0
radius_mean                0.0
texture_mean               0.0
perimeter_mean             0.0
area_mean                  0.0
smoothness_mean            0.0
compactness_mean           0.0
concavity_mean             0.0
concave points_mean        0.0
symmetry_mean              0.0
fractal_dimension_mean     0.0
radius_se                  0.0
texture_se                 0.0
perimeter_se               0.0
area_se                    0.0
smoothness_se              0.0
compactness_se             0.0
concavity_se               0.0
concave points_se          0.0
symmetry_se                0.0
fractal_dimension_se       0.0
radius_worst               0.0
texture_worst              0.0
perimeter_worst            0.0
area_worst                 0.0
smoothness_worst           0.0
compactness_worst          0.0
concavity_worst            0.0
concave points_worst       0.0
symmetry_worst             0.0
fractal_dimension_worst    0.0
Unnamed:

In [23]:
df = df.drop(columns=['id','Unnamed: 32'])

In [24]:
df["diagnosis"].unique()

<StringArray>
['M', 'B']
Length: 2, dtype: str

In [25]:
## Je vais encoder les valeurs "M" et "B". M par 1 et B par 0
df["diagnosis"].replace({"M":1,"B":0}).astype(int)

0      1
1      1
2      1
3      1
4      1
      ..
564    1
565    1
566    1
567    1
568    0
Name: diagnosis, Length: 569, dtype: int64

In [26]:
## Je sépare mes valeurs en données explicatives et en résultats
y = df["diagnosis"]
df = df.drop('diagnosis',axis=1)
X = df

In [27]:
## Je vais générer des nombres aléatoires qui represente les paramètres du model et le biais
def init(X):
    w = np.random.randn(X.shape[1])## les parametres represente les catégories explicatives
    b = np.random.randn()
    return w,b

In [28]:
def model(X,w,b):
    Z = X@w+b ## Je calcule un score en fonction des variables
    A = 1/(1+np.exp(-Z)) # Je le convertis en probabilité
    return A

In [29]:
def cout(A,X,y):## Je calcule la fonction cout. J'utilise la log_loss function
    loss = (-1/X.shape[0])*(np.dot(y,np.log(A))+np.dot((1-y),np.log(1-A)))
    return loss
    

In [30]:
def gradient(A,X,y):
    dw = (1/X.shape[0])*X.T@(A-y)
    db = (1/X.shape[0])*np.sum(A-y)
    return dw,db

In [31]:
def MAJ(w,b,dw,db,learning_rate):
    w = w - learning_rate*dw
    b = b - learning_rate*db
    return w,b

In [34]:
## l'entrainement
def fit(X,y,learning_rate = 0.1,iter = 100):
    w,b = init(X)
    for i in range(iter):
        A = model (X,w,b)
        #loss = cout(A,X,y)
        dw,db = gradient(A,X,y)
        MAJ(w,b,dw,db,learning_rate)
        

In [35]:
fit(X,y)

TypeError: unsupported operand type(s) for -: 'float' and 'str'